In [0]:
%sql
BEGIN
  DECLARE v_period_end_date DATE;
  DECLARE v_rfm_max         DATE;

  SET v_period_end_date = (
    SELECT last_day(add_months(MAX(period_end_date), 1))
    FROM data_exploration_hub.marketing_hub.mkt_dashboard_visitation_behavior
  );
  SET v_rfm_max = (
    SELECT to_date(CAST(MAX(SnapshotDateId) AS STRING), 'yyyyMMdd')
    FROM cpx_dataanalytics_gold.customer360.c360_f_rfm_model
  );

  WHILE date_add(v_period_end_date, 1) <= v_rfm_max
        AND v_period_end_date <= date_sub(current_date(), 1) DO
    DELETE FROM data_exploration_hub.marketing_hub.mkt_dashboard_visitation_behavior
    WHERE period_end_date = v_period_end_date;

    INSERT INTO data_exploration_hub.marketing_hub.mkt_dashboard_visitation_behavior
    WITH params AS (
        SELECT v_period_end_date AS period_end_date
    ),

    periods AS (
        SELECT 'monthly'   AS period_type,
               CAST(DATE_TRUNC('MONTH', period_end_date) AS DATE) AS start_date,
               period_end_date AS end_date
        FROM params
        UNION ALL
        SELECT 'ptd',
               CAST(DATE'2021-08-01' AS DATE),
               period_end_date
        FROM params
        UNION ALL
        SELECT 'p12m',
               CAST(ADD_MONTHS(DATE_TRUNC('MONTH', period_end_date), -11) AS DATE),
               period_end_date
        FROM params
        UNION ALL
        SELECT 'ytd',
               CAST(DATE_TRUNC('YEAR', period_end_date) AS DATE),
               period_end_date
        FROM params
        UNION ALL
        SELECT 'quarterly',
               CAST(ADD_MONTHS(DATE_TRUNC('MONTH', period_end_date), -2) AS DATE),
               period_end_date
        FROM params
    ),

    rfm_snapshot AS (
        SELECT CDE_ID, Segment
        FROM cpx_dataanalytics_gold.customer360.c360_f_rfm_model, params p
        WHERE SnapshotDateId = date_format(DATE_ADD(p.period_end_date, 1), 'yyyyMMdd')
    ),

    exh_cc_visits AS (
        SELECT
            t.CDE_ID AS cde_id,
            d.datefull,
            SUM(CASE WHEN t.TicketFLAG = 'Yes' THEN t.Quantity ELSE 0 END)     AS ticket_qty,
            SUM(CASE WHEN t.TicketFLAG = 'Yes' THEN t.Net_Revenue ELSE 0 END) AS ticket_spend,
            SUM(CASE WHEN t.ConcessionFLAG = 'Yes' THEN t.Quantity ELSE 0 END)     AS concession_qty,
            SUM(CASE WHEN t.ConcessionFLAG = 'Yes' THEN t.Net_Revenue ELSE 0 END) AS concession_spend,
            CASE WHEN SUM(t.Quantity) > 0 THEN 1 ELSE 0 END AS is_visit
        FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
        JOIN cpx_dataanalytics_gold.customer360.cpx_d_location loc ON t.LocationId = loc.LocationID
        JOIN cpx_dataanalytics_gold.customer360.cpx_d_date d ON t.VisitDateId = d.dateid
        CROSS JOIN params p
        WHERE loc.LC_Location_Type_Description = 'Cineplex Theatre'
          AND t.CDE_ID IS NOT NULL
          AND t.CineClubFLAG = 'Yes'
          AND d.datefull BETWEEN DATE'2021-08-01' AND p.period_end_date
        GROUP BY t.CDE_ID, d.datefull
    ),

    exh_scene_visits AS (
        SELECT
            t.CDE_ID AS cde_id,
            d.datefull,
            SUM(CASE WHEN t.TicketFLAG = 'Yes' THEN t.Quantity ELSE 0 END)     AS ticket_qty,
            SUM(CASE WHEN t.TicketFLAG = 'Yes' THEN t.Net_Revenue ELSE 0 END) AS ticket_spend,
            SUM(CASE WHEN t.ConcessionFLAG = 'Yes' THEN t.Quantity ELSE 0 END)     AS concession_qty,
            SUM(CASE WHEN t.ConcessionFLAG = 'Yes' THEN t.Net_Revenue ELSE 0 END) AS concession_spend,
            CASE WHEN SUM(t.Quantity) > 0 THEN 1 ELSE 0 END AS is_visit
        FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
        JOIN cpx_dataanalytics_gold.customer360.cpx_d_location loc ON t.LocationId = loc.LocationID
        JOIN cpx_dataanalytics_gold.customer360.cpx_d_date d ON t.VisitDateId = d.dateid
        CROSS JOIN params p
        WHERE loc.LC_Location_Type_Description = 'Cineplex Theatre'
          AND t.CDE_ID IS NOT NULL
          AND t.SCENEFLAG = 'Yes'
          AND d.datefull BETWEEN DATE'2021-08-01' AND p.period_end_date
        GROUP BY t.CDE_ID, d.datefull
    ),

    lbe_cc_visits AS (
        SELECT
            t.CDE_ID AS cde_id,
            d.datefull,
            SUM(CASE WHEN i.Item_Class_Description1 IN (
                'TRR - AMUSEMENT - GAMES',
                'TRR - AMUSEMENT - PHYSICAL GAMES','TRR - PROMO - AMUSEMENT'
            ) THEN t.Net_Revenue ELSE 0 END) AS gaming_spend,
            SUM(CASE WHEN i.Item_Class_Description1 IN (
                'PDM - MENU FOOD - CANDY','TRR - INV ALC BEV - BOTTLE','TRR - INV ALC BEV - CAN',
                'TRR - INV ALC BEV - WINE','TRR - MENU ALC BEV - COCKTAIL','TRR - MENU ALC BEV - DRAUGHT BEER',
                'TRR - MENU BEV - NA COLD BEV','TRR - MENU BEV - NA HOT BEV','TRR - MENU FOOD - APP',
                'TRR - MENU FOOD - DESSERT','TRR - MENU FOOD - KIDS','TRR - MENU FOOD - PIZZA',
                'TRR - MENU FOOD - POUTINE','TRR - MENU FOOD - SANDWICHES','TRR - MOD MENU - ADD-ON',
                'TRR - INV BEV - NA BEV','TRR - MENU ALC BEV - LIQUOR','TRR - MENU ALC BEV - WINE',
                'TRR - MENU FOOD - ENTRÉE','TRR - MENU FOOD - SALAD','VHO-LICENSED FOOD',
                'TRR - INV ALC BEV - LIQUOR','VHO-CANDY','VIP Food - Addons','VHO-LICENSED DRINKS',
                'TRR - MENU FOOD - BRUNCH','VHO-DRINK'
            ) THEN t.Net_Revenue ELSE 0 END) AS fnb_spend
        FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
        JOIN cpx_dataanalytics_gold.customer360.cpx_d_location loc ON t.LocationId = loc.LocationID
        JOIN cpx_dataanalytics_gold.customer360.cpx_d_item i ON t.ItemSkey = i.itemskey
        JOIN cpx_dataanalytics_gold.customer360.cpx_d_date d ON t.VisitDateId = d.dateid
        CROSS JOIN params p
        WHERE loc.LC_Location_Type_Description LIKE 'Restaurant%'
          AND loc.LC_Trade_Brand_Description IN ('Rec Room','Playdium')
          AND t.CDE_ID IS NOT NULL
          AND t.CineClubFLAG = 'Yes'
          AND d.datefull BETWEEN DATE'2021-08-01' AND p.period_end_date
        GROUP BY t.CDE_ID, d.datefull
    ),

    lbe_scene_visits AS (
        SELECT
            t.CDE_ID AS cde_id,
            d.datefull,
            SUM(CASE WHEN i.Item_Class_Description1 IN (
                'TRR - AMUSEMENT - GAMES',
                'TRR - AMUSEMENT - PHYSICAL GAMES','TRR - PROMO - AMUSEMENT'
            ) THEN t.Net_Revenue ELSE 0 END) AS gaming_spend,
            SUM(CASE WHEN i.Item_Class_Description1 IN (
                'PDM - MENU FOOD - CANDY','TRR - INV ALC BEV - BOTTLE','TRR - INV ALC BEV - CAN',
                'TRR - INV ALC BEV - WINE','TRR - MENU ALC BEV - COCKTAIL','TRR - MENU ALC BEV - DRAUGHT BEER',
                'TRR - MENU BEV - NA COLD BEV','TRR - MENU BEV - NA HOT BEV','TRR - MENU FOOD - APP',
                'TRR - MENU FOOD - DESSERT','TRR - MENU FOOD - KIDS','TRR - MENU FOOD - PIZZA',
                'TRR - MENU FOOD - POUTINE','TRR - MENU FOOD - SANDWICHES','TRR - MOD MENU - ADD-ON',
                'TRR - INV BEV - NA BEV','TRR - MENU ALC BEV - LIQUOR','TRR - MENU ALC BEV - WINE',
                'TRR - MENU FOOD - ENTRÉE','TRR - MENU FOOD - SALAD','VHO-LICENSED FOOD',
                'TRR - INV ALC BEV - LIQUOR','VHO-CANDY','VIP Food - Addons','VHO-LICENSED DRINKS',
                'TRR - MENU FOOD - BRUNCH','VHO-DRINK'
            ) THEN t.Net_Revenue ELSE 0 END) AS fnb_spend
        FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
        JOIN cpx_dataanalytics_gold.customer360.cpx_d_location loc ON t.LocationId = loc.LocationID
        JOIN cpx_dataanalytics_gold.customer360.cpx_d_item i ON t.ItemSkey = i.itemskey
        JOIN cpx_dataanalytics_gold.customer360.cpx_d_date d ON t.VisitDateId = d.dateid
        CROSS JOIN params p
        WHERE loc.LC_Location_Type_Description LIKE 'Restaurant%'
          AND loc.LC_Trade_Brand_Description IN ('Rec Room','Playdium')
          AND t.CDE_ID IS NOT NULL
          AND t.SCENEFLAG = 'Yes'
          AND d.datefull BETWEEN DATE'2021-08-01' AND p.period_end_date
        GROUP BY t.CDE_ID, d.datefull
    ),

    exh_cc_exploded AS (
        SELECT pr.period_type, pr.start_date AS period_start_date, pr.end_date AS period_end_date,
               v.cde_id, v.ticket_qty, v.ticket_spend, v.concession_qty, v.concession_spend, v.is_visit,
               MAX(v.is_visit) OVER (PARTITION BY pr.period_type, v.cde_id) AS member_has_visit,
               COALESCE(r.Segment, 'Dormant') AS rfm_category
        FROM exh_cc_visits v
        JOIN periods pr ON v.datefull BETWEEN pr.start_date AND pr.end_date
        LEFT JOIN rfm_snapshot r ON CAST(v.cde_id AS INT) = r.CDE_ID
    ),
    exh_scene_exploded AS (
        SELECT pr.period_type, pr.start_date AS period_start_date, pr.end_date AS period_end_date,
               v.cde_id, v.ticket_qty, v.ticket_spend, v.concession_qty, v.concession_spend, v.is_visit,
               MAX(v.is_visit) OVER (PARTITION BY pr.period_type, v.cde_id) AS member_has_visit,
               COALESCE(r.Segment, 'Dormant') AS rfm_category
        FROM exh_scene_visits v
        JOIN periods pr ON v.datefull BETWEEN pr.start_date AND pr.end_date
        LEFT JOIN rfm_snapshot r ON CAST(v.cde_id AS INT) = r.CDE_ID
    ),
    lbe_cc_exploded AS (
        SELECT pr.period_type, pr.start_date AS period_start_date, pr.end_date AS period_end_date,
               v.cde_id, v.gaming_spend, v.fnb_spend,
               COALESCE(r.Segment, 'Dormant') AS rfm_category
        FROM lbe_cc_visits v
        JOIN periods pr ON v.datefull BETWEEN pr.start_date AND pr.end_date
        LEFT JOIN rfm_snapshot r ON CAST(v.cde_id AS INT) = r.CDE_ID
    ),
    lbe_scene_exploded AS (
        SELECT pr.period_type, pr.start_date AS period_start_date, pr.end_date AS period_end_date,
               v.cde_id, v.gaming_spend, v.fnb_spend,
               COALESCE(r.Segment, 'Dormant') AS rfm_category
        FROM lbe_scene_visits v
        JOIN periods pr ON v.datefull BETWEEN pr.start_date AND pr.end_date
        LEFT JOIN rfm_snapshot r ON CAST(v.cde_id AS INT) = r.CDE_ID
    ),

    exh_cc_all AS (
        SELECT period_type, period_start_date, period_end_date, 'All' AS rfm_segment,
            COUNT(DISTINCT CASE WHEN is_visit = 1 THEN cde_id END) AS distinct_visitors_cc,
            SUM(is_visit) AS total_visits_cc,
            ROUND(SUM(CASE WHEN member_has_visit = 1 THEN ticket_qty ELSE 0 END) * 1.0 / COUNT(DISTINCT CASE WHEN is_visit = 1 THEN cde_id END), 2) AS avg_tickets_per_member_cc,
            ROUND(SUM(is_visit) * 1.0 / COUNT(DISTINCT CASE WHEN is_visit = 1 THEN cde_id END), 2) AS avg_visits_per_member_cc,
            ROUND(SUM(CASE WHEN member_has_visit = 1 THEN ticket_spend ELSE 0 END) / COUNT(DISTINCT CASE WHEN is_visit = 1 THEN cde_id END), 2) AS avg_ticket_spend_per_member_cc,
            ROUND(SUM(CASE WHEN member_has_visit = 1 THEN concession_spend ELSE 0 END) / COUNT(DISTINCT CASE WHEN is_visit = 1 THEN cde_id END), 2) AS avg_concession_spend_per_member_cc,
            ROUND(SUM(CASE WHEN concession_qty > 0 AND is_visit = 1 THEN 1 ELSE 0 END) * 100.0 / NULLIF(SUM(is_visit), 0), 2) AS concession_incidence_cc
        FROM exh_cc_exploded
        GROUP BY period_type, period_start_date, period_end_date
    ),
    exh_cc_top3 AS (
        SELECT period_type, period_start_date, period_end_date, 'Top 3' AS rfm_segment,
            COUNT(DISTINCT CASE WHEN is_visit = 1 THEN cde_id END) AS distinct_visitors_cc,
            SUM(is_visit) AS total_visits_cc,
            ROUND(SUM(CASE WHEN member_has_visit = 1 THEN ticket_qty ELSE 0 END) * 1.0 / COUNT(DISTINCT CASE WHEN is_visit = 1 THEN cde_id END), 2) AS avg_tickets_per_member_cc,
            ROUND(SUM(is_visit) * 1.0 / COUNT(DISTINCT CASE WHEN is_visit = 1 THEN cde_id END), 2) AS avg_visits_per_member_cc,
            ROUND(SUM(CASE WHEN member_has_visit = 1 THEN ticket_spend ELSE 0 END) / COUNT(DISTINCT CASE WHEN is_visit = 1 THEN cde_id END), 2) AS avg_ticket_spend_per_member_cc,
            ROUND(SUM(CASE WHEN member_has_visit = 1 THEN concession_spend ELSE 0 END) / COUNT(DISTINCT CASE WHEN is_visit = 1 THEN cde_id END), 2) AS avg_concession_spend_per_member_cc,
            ROUND(SUM(CASE WHEN concession_qty > 0 AND is_visit = 1 THEN 1 ELSE 0 END) * 100.0 / NULLIF(SUM(is_visit), 0), 2) AS concession_incidence_cc
        FROM exh_cc_exploded
        WHERE rfm_category IN ('Champions', 'Potential Loyalist', 'Value Conscious Loyals')
        GROUP BY period_type, period_start_date, period_end_date
    ),
    exh_cc_by_segment AS (
        SELECT period_type, period_start_date, period_end_date, rfm_category AS rfm_segment,
            COUNT(DISTINCT CASE WHEN is_visit = 1 THEN cde_id END) AS distinct_visitors_cc,
            SUM(is_visit) AS total_visits_cc,
            ROUND(SUM(CASE WHEN member_has_visit = 1 THEN ticket_qty ELSE 0 END) * 1.0 / COUNT(DISTINCT CASE WHEN is_visit = 1 THEN cde_id END), 2) AS avg_tickets_per_member_cc,
            ROUND(SUM(is_visit) * 1.0 / COUNT(DISTINCT CASE WHEN is_visit = 1 THEN cde_id END), 2) AS avg_visits_per_member_cc,
            ROUND(SUM(CASE WHEN member_has_visit = 1 THEN ticket_spend ELSE 0 END) / COUNT(DISTINCT CASE WHEN is_visit = 1 THEN cde_id END), 2) AS avg_ticket_spend_per_member_cc,
            ROUND(SUM(CASE WHEN member_has_visit = 1 THEN concession_spend ELSE 0 END) / COUNT(DISTINCT CASE WHEN is_visit = 1 THEN cde_id END), 2) AS avg_concession_spend_per_member_cc,
            ROUND(SUM(CASE WHEN concession_qty > 0 AND is_visit = 1 THEN 1 ELSE 0 END) * 100.0 / NULLIF(SUM(is_visit), 0), 2) AS concession_incidence_cc
        FROM exh_cc_exploded
        GROUP BY period_type, period_start_date, period_end_date, rfm_category
    ),
    exh_cc_final AS (
        SELECT * FROM exh_cc_all UNION ALL SELECT * FROM exh_cc_top3 UNION ALL SELECT * FROM exh_cc_by_segment
    ),

    exh_scene_all AS (
        SELECT period_type, period_start_date, period_end_date, 'All' AS rfm_segment,
            COUNT(DISTINCT CASE WHEN is_visit = 1 THEN cde_id END) AS distinct_visitors_scene,
            SUM(is_visit) AS total_visits_scene,
            ROUND(SUM(CASE WHEN member_has_visit = 1 THEN ticket_qty ELSE 0 END) * 1.0 / COUNT(DISTINCT CASE WHEN is_visit = 1 THEN cde_id END), 2) AS avg_tickets_per_member_scene,
            ROUND(SUM(is_visit) * 1.0 / COUNT(DISTINCT CASE WHEN is_visit = 1 THEN cde_id END), 2) AS avg_visits_per_member_scene,
            ROUND(SUM(CASE WHEN member_has_visit = 1 THEN ticket_spend ELSE 0 END) / COUNT(DISTINCT CASE WHEN is_visit = 1 THEN cde_id END), 2) AS avg_ticket_spend_per_member_scene,
            ROUND(SUM(CASE WHEN member_has_visit = 1 THEN concession_spend ELSE 0 END) / COUNT(DISTINCT CASE WHEN is_visit = 1 THEN cde_id END), 2) AS avg_concession_spend_per_member_scene,
            ROUND(SUM(CASE WHEN concession_qty > 0 AND is_visit = 1 THEN 1 ELSE 0 END) * 100.0 / NULLIF(SUM(is_visit), 0), 2) AS concession_incidence_scene
        FROM exh_scene_exploded
        GROUP BY period_type, period_start_date, period_end_date
    ),
    exh_scene_top3 AS (
        SELECT period_type, period_start_date, period_end_date, 'Top 3' AS rfm_segment,
            COUNT(DISTINCT CASE WHEN is_visit = 1 THEN cde_id END) AS distinct_visitors_scene,
            SUM(is_visit) AS total_visits_scene,
            ROUND(SUM(CASE WHEN member_has_visit = 1 THEN ticket_qty ELSE 0 END) * 1.0 / COUNT(DISTINCT CASE WHEN is_visit = 1 THEN cde_id END), 2) AS avg_tickets_per_member_scene,
            ROUND(SUM(is_visit) * 1.0 / COUNT(DISTINCT CASE WHEN is_visit = 1 THEN cde_id END), 2) AS avg_visits_per_member_scene,
            ROUND(SUM(CASE WHEN member_has_visit = 1 THEN ticket_spend ELSE 0 END) / COUNT(DISTINCT CASE WHEN is_visit = 1 THEN cde_id END), 2) AS avg_ticket_spend_per_member_scene,
            ROUND(SUM(CASE WHEN member_has_visit = 1 THEN concession_spend ELSE 0 END) / COUNT(DISTINCT CASE WHEN is_visit = 1 THEN cde_id END), 2) AS avg_concession_spend_per_member_scene,
            ROUND(SUM(CASE WHEN concession_qty > 0 AND is_visit = 1 THEN 1 ELSE 0 END) * 100.0 / NULLIF(SUM(is_visit), 0), 2) AS concession_incidence_scene
        FROM exh_scene_exploded
        WHERE rfm_category IN ('Champions', 'Potential Loyalist', 'Value Conscious Loyals')
        GROUP BY period_type, period_start_date, period_end_date
    ),
    exh_scene_by_segment AS (
        SELECT period_type, period_start_date, period_end_date, rfm_category AS rfm_segment,
            COUNT(DISTINCT CASE WHEN is_visit = 1 THEN cde_id END) AS distinct_visitors_scene,
            SUM(is_visit) AS total_visits_scene,
            ROUND(SUM(CASE WHEN member_has_visit = 1 THEN ticket_qty ELSE 0 END) * 1.0 / COUNT(DISTINCT CASE WHEN is_visit = 1 THEN cde_id END), 2) AS avg_tickets_per_member_scene,
            ROUND(SUM(is_visit) * 1.0 / COUNT(DISTINCT CASE WHEN is_visit = 1 THEN cde_id END), 2) AS avg_visits_per_member_scene,
            ROUND(SUM(CASE WHEN member_has_visit = 1 THEN ticket_spend ELSE 0 END) / COUNT(DISTINCT CASE WHEN is_visit = 1 THEN cde_id END), 2) AS avg_ticket_spend_per_member_scene,
            ROUND(SUM(CASE WHEN member_has_visit = 1 THEN concession_spend ELSE 0 END) / COUNT(DISTINCT CASE WHEN is_visit = 1 THEN cde_id END), 2) AS avg_concession_spend_per_member_scene,
            ROUND(SUM(CASE WHEN concession_qty > 0 AND is_visit = 1 THEN 1 ELSE 0 END) * 100.0 / NULLIF(SUM(is_visit), 0), 2) AS concession_incidence_scene
        FROM exh_scene_exploded
        GROUP BY period_type, period_start_date, period_end_date, rfm_category
    ),
    exh_scene_final AS (
        SELECT * FROM exh_scene_all UNION ALL SELECT * FROM exh_scene_top3 UNION ALL SELECT * FROM exh_scene_by_segment
    ),

    lbe_cc_all AS (
        SELECT period_type, period_start_date, period_end_date, 'All' AS rfm_segment,
            COUNT(DISTINCT cde_id) AS distinct_visitors_cc,
            COUNT(*) AS total_visits_cc,
            ROUND(SUM(gaming_spend) / COUNT(DISTINCT cde_id), 2) AS avg_gaming_spend_per_member_cc,
            ROUND(SUM(fnb_spend) / COUNT(DISTINCT cde_id), 2)    AS avg_fnb_spend_per_member_cc,
            ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT cde_id), 2)    AS avg_visits_per_member_cc
        FROM lbe_cc_exploded
        GROUP BY period_type, period_start_date, period_end_date
    ),
    lbe_cc_top3 AS (
        SELECT period_type, period_start_date, period_end_date, 'Top 3' AS rfm_segment,
            COUNT(DISTINCT cde_id) AS distinct_visitors_cc,
            COUNT(*) AS total_visits_cc,
            ROUND(SUM(gaming_spend) / COUNT(DISTINCT cde_id), 2) AS avg_gaming_spend_per_member_cc,
            ROUND(SUM(fnb_spend) / COUNT(DISTINCT cde_id), 2)    AS avg_fnb_spend_per_member_cc,
            ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT cde_id), 2)    AS avg_visits_per_member_cc
        FROM lbe_cc_exploded
        WHERE rfm_category IN ('Champions', 'Potential Loyalist', 'Value Conscious Loyals')
        GROUP BY period_type, period_start_date, period_end_date
    ),
    lbe_cc_by_segment AS (
        SELECT period_type, period_start_date, period_end_date, rfm_category AS rfm_segment,
            COUNT(DISTINCT cde_id) AS distinct_visitors_cc,
            COUNT(*) AS total_visits_cc,
            ROUND(SUM(gaming_spend) / COUNT(DISTINCT cde_id), 2) AS avg_gaming_spend_per_member_cc,
            ROUND(SUM(fnb_spend) / COUNT(DISTINCT cde_id), 2)    AS avg_fnb_spend_per_member_cc,
            ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT cde_id), 2)    AS avg_visits_per_member_cc
        FROM lbe_cc_exploded
        GROUP BY period_type, period_start_date, period_end_date, rfm_category
    ),
    lbe_cc_final AS (
        SELECT * FROM lbe_cc_all UNION ALL SELECT * FROM lbe_cc_top3 UNION ALL SELECT * FROM lbe_cc_by_segment
    ),

    lbe_scene_all AS (
        SELECT period_type, period_start_date, period_end_date, 'All' AS rfm_segment,
            COUNT(DISTINCT cde_id) AS distinct_visitors_scene,
            COUNT(*) AS total_visits_scene,
            ROUND(SUM(gaming_spend) / COUNT(DISTINCT cde_id), 2) AS avg_gaming_spend_per_member_scene,
            ROUND(SUM(fnb_spend) / COUNT(DISTINCT cde_id), 2)    AS avg_fnb_spend_per_member_scene,
            ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT cde_id), 2)    AS avg_visits_per_member_scene
        FROM lbe_scene_exploded
        GROUP BY period_type, period_start_date, period_end_date
    ),
    lbe_scene_top3 AS (
        SELECT period_type, period_start_date, period_end_date, 'Top 3' AS rfm_segment,
            COUNT(DISTINCT cde_id) AS distinct_visitors_scene,
            COUNT(*) AS total_visits_scene,
            ROUND(SUM(gaming_spend) / COUNT(DISTINCT cde_id), 2) AS avg_gaming_spend_per_member_scene,
            ROUND(SUM(fnb_spend) / COUNT(DISTINCT cde_id), 2)    AS avg_fnb_spend_per_member_scene,
            ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT cde_id), 2)    AS avg_visits_per_member_scene
        FROM lbe_scene_exploded
        WHERE rfm_category IN ('Champions', 'Potential Loyalist', 'Value Conscious Loyals')
        GROUP BY period_type, period_start_date, period_end_date
    ),
    lbe_scene_by_segment AS (
        SELECT period_type, period_start_date, period_end_date, rfm_category AS rfm_segment,
            COUNT(DISTINCT cde_id) AS distinct_visitors_scene,
            COUNT(*) AS total_visits_scene,
            ROUND(SUM(gaming_spend) / COUNT(DISTINCT cde_id), 2) AS avg_gaming_spend_per_member_scene,
            ROUND(SUM(fnb_spend) / COUNT(DISTINCT cde_id), 2)    AS avg_fnb_spend_per_member_scene,
            ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT cde_id), 2)    AS avg_visits_per_member_scene
        FROM lbe_scene_exploded
        GROUP BY period_type, period_start_date, period_end_date, rfm_category
    ),
    lbe_scene_final AS (
        SELECT * FROM lbe_scene_all UNION ALL SELECT * FROM lbe_scene_top3 UNION ALL SELECT * FROM lbe_scene_by_segment
    ),

    exh_rows AS (
        SELECT
            c.period_type, c.period_start_date, c.period_end_date, 'EXH' AS venue_type, c.rfm_segment,
            c.distinct_visitors_cc, s.distinct_visitors_scene,
            c.total_visits_cc, s.total_visits_scene,
            c.avg_tickets_per_member_cc, s.avg_tickets_per_member_scene,
            c.avg_visits_per_member_cc, s.avg_visits_per_member_scene,
            c.avg_ticket_spend_per_member_cc, s.avg_ticket_spend_per_member_scene,
            c.avg_concession_spend_per_member_cc, s.avg_concession_spend_per_member_scene,
            CAST(NULL AS DECIMAL(10,2)) AS avg_gaming_spend_per_member_cc,
            CAST(NULL AS DECIMAL(10,2)) AS avg_gaming_spend_per_member_scene,
            CAST(NULL AS DECIMAL(10,2)) AS avg_fnb_spend_per_member_cc,
            CAST(NULL AS DECIMAL(10,2)) AS avg_fnb_spend_per_member_scene,
            c.concession_incidence_cc, s.concession_incidence_scene
        FROM exh_cc_final c
        JOIN exh_scene_final s
            ON c.period_type = s.period_type AND c.rfm_segment = s.rfm_segment
    ),
    lbe_rows AS (
        SELECT
            c.period_type, c.period_start_date, c.period_end_date, 'LBE' AS venue_type, c.rfm_segment,
            c.distinct_visitors_cc, s.distinct_visitors_scene,
            c.total_visits_cc, s.total_visits_scene,
            CAST(NULL AS DECIMAL(10,2)) AS avg_tickets_per_member_cc,
            CAST(NULL AS DECIMAL(10,2)) AS avg_tickets_per_member_scene,
            c.avg_visits_per_member_cc, s.avg_visits_per_member_scene,
            CAST(NULL AS DECIMAL(10,2)) AS avg_ticket_spend_per_member_cc,
            CAST(NULL AS DECIMAL(10,2)) AS avg_ticket_spend_per_member_scene,
            CAST(NULL AS DECIMAL(10,2)) AS avg_concession_spend_per_member_cc,
            CAST(NULL AS DECIMAL(10,2)) AS avg_concession_spend_per_member_scene,
            c.avg_gaming_spend_per_member_cc, s.avg_gaming_spend_per_member_scene,
            c.avg_fnb_spend_per_member_cc, s.avg_fnb_spend_per_member_scene,
            CAST(NULL AS DECIMAL(10,2)) AS concession_incidence_cc,
            CAST(NULL AS DECIMAL(10,2)) AS concession_incidence_scene
        FROM lbe_cc_final c
        JOIN lbe_scene_final s
            ON c.period_type = s.period_type AND c.rfm_segment = s.rfm_segment
    )

    SELECT *, CAST(DATE_TRUNC('MONTH', period_end_date) AS DATE) AS report_month FROM exh_rows
    UNION ALL
    SELECT *, CAST(DATE_TRUNC('MONTH', period_end_date) AS DATE) AS report_month FROM lbe_rows
    ORDER BY
        CASE period_type WHEN 'ptd' THEN 0 WHEN 'p12m' THEN 1 WHEN 'ytd' THEN 2 WHEN 'quarterly' THEN 3 WHEN 'monthly' THEN 4 END,
        venue_type,
        CASE rfm_segment
            WHEN 'All' THEN 0 WHEN 'Top 3' THEN 1
            WHEN 'Champions' THEN 2 WHEN 'Potential Loyalist' THEN 3
            WHEN 'Value Conscious Loyals' THEN 4 WHEN 'Promising' THEN 5
            WHEN 'At Risk' THEN 6 WHEN 'New' THEN 7
            WHEN 'Fleeting' THEN 8 WHEN 'High Value Lapsed' THEN 9
            WHEN 'Dormant' THEN 10
        END;

    SET v_period_end_date = last_day(add_months(v_period_end_date, 1));
  END WHILE;
END;